# Setup

In [1]:
import numpy as np, timeit, time, matplotlib.pyplot as plt, json, os
from tqdm import tqdm

In [2]:
from binary_models import *
from benchmark_models import get_SMK_SCM
from benchmark_models import nSMK_model, get_avg_nSMK_SCM, get_lucb_nSMK_SCM
from evaluation import evaluate_full
# from actualcauses import beam_search, show_rules, iterative_identification
# from actualcauses.base_algorithm import *
from actualcauses_local.base_algorithm import beam_search, get_rules, get_initial_rules
from actualcauses_local.lucb import lucb
from actualcauses_local.iterative_subinstance_algorithm import iterative_identification as isi
from general import *

In [3]:
from experiments import run_noisy_SMK

# Functions

In [4]:
def var2label(S, variables):
    return {variables[i] for i in S}

In [5]:
# def sort_key(rule_values):
#     _, rule_output, rule_score, C, W, _ = rule_values
#     return (rule_score, len(C), C, len(W), W)

In [6]:
# def get_next_beams(non_causes, beam_size, Cs):
#     non_causes = [v for v in non_causes if not any([c <= v[3] for c in Cs])]
#     # Score and sort the remaining
#     non_causes = sorted(non_causes, key=sort_key)
#     # Filter the top-b
#     if beam_size != -1:
#         non_causes = non_causes[:beam_size]
#     # Keep only the interventions to build next ones
#     beams = [rule_value[0] for rule_value in non_causes]
#     return beams

In [7]:
# def beam_search(
#     instance, domains, simulation, variables, # SCM
#     max_steps=5, beam_size=10, epsilon=.05, early_stop=True, max_time=None, # Parameters
#     var_mapping=None, ref_w=tuple(), Cs=None, # Additional parameters when running for sub-instance
#     verbose=0, 
#     ):
#     # verbose: 
#     #  = 1 -> best cause at the end, tqdm for steps
#     #  >= 2 -> removes step tqdm, adds step header + number of cause found + best and worse non causes
#     #  >= 3 -> adds all causes + tqdm for get_rules
    
#     all_causes = []
#     init_time = time.time()
#     if Cs is None: Cs = []
#     if max_steps == -1 or max_steps is None: iterator = count(start=1, step=1)
#     else: iterator = range(1,max_steps+1)

#     for t in tqdm(iterator, disable=(verbose!=1)):
#         # Render the step
#         if verbose >= 2: print(f"{f'Step {t}':=^30}")
            
#         # Create the rules for step t base on the ones from t-1, we use the initial ones if t==1
#         beams = get_rules(beams if t > 1 else None, domains, instance, Cs, verbose >= 3)

#         # Check for early stop
#         if check_early_stop(beams, early_stop, all_causes, max_time, init_time):
#             break
            
#         # Render how many nodes will be evaluated
#         if verbose >= 2: print(f"Evaluating {len(beams)} rules")
            
#         # Evaluate the rules using the simulation 
#         sim_beams = map_beams(beams, var_mapping, ref_w)
#         cf_values = simulation(sim_beams)
        
#         # Build the tuples of rule values
#         causes, non_causes = split_rules(beams, cf_values, instance, epsilon)

#         # Filter causes to keep only minimal ones and save them
#         causes = filter_minimality(causes)
#         all_causes += causes
#         for rule_value in causes:
#             Cs.append(rule_value[3])

#         ### Build next beams
#         beams = get_next_beams(non_causes, beam_size, Cs)
        
#         # Render step output
#         render_step(verbose, causes, non_causes, instance, variables)

#     # Sort final rule set
#     all_causes.sort(key=sort_key)

#     # Render final result
#     if verbose:
#         print(f"----> Found {len(all_causes)} causes.")
#         if all_causes:
#             print(f"{'Overall best rule:':=^30}")
#             show_rule(all_causes[0], variables)
        
#     return all_causes

In [8]:
# This file is inspired by https://github.com/marcotcr/anchor

# def kl_bernoulli(p, q):
#     p = min(0.9999999999999999, max(0.0000001, p))
#     q = min(0.9999999999999999, max(0.0000001, q))
#     return (p * np.log(float(p) / q) + (1 - p) *
#             np.log(float(1 - p) / (1 - q)))

# def dup_bernoulli(p, level):
#     lm = p
#     um = min(min(1, p + np.sqrt(level / 2.)), 1)
#     qm = (um + lm) / 2.
#     if kl_bernoulli(p, qm) > level:
#         um = qm
#     else:
#         lm = qm
#     return um

# def dlow_bernoulli(p, level):
#     um = p
#     lm = max(min(1, p - np.sqrt(level / 2.)), 0)
#     qm = (um + lm) / 2.
#     if kl_bernoulli(p, qm) > level:
#         lm = qm
#     else:
#         um = qm
#     return lm

# def compute_beta(n_features, t):
#     delta = .1
#     alpha = 1.1
#     k = 405.5
#     temp = np.log(k * n_features * (t ** alpha) / delta)
#     return temp + np.log(temp)


In [9]:
# def lucb(simulation, rules, beam_size, a=.05, beam_eps=.1, cause_eps=.01, non_cause_esp=.01, 
#          max_iter=200, verbose=0, batch_size=10, init_batch_size=20, lucb_infos=None):
        
#     n_arms = len(rules) # Doing armed bandits with the rules to evaluate
#     # For each arm we keep track of the number of samples, the number of success, the upper bound and the lower bound
#     positives, scores, lb, means, mean_scores = np.zeros((5, n_arms))
#     ub = np.ones(n_arms)
#     score_ub, score_lb = np.ones(n_arms), np.zeros(n_arms)
#     n_samples = np.zeros(n_arms, dtype=int)
#     beta = 0
    
#     # Utils function
#     def action_arm(arm, init=False):
#         bs = init_batch_size if init else batch_size
#         for _ in range(bs):
#             _, label, score = simulation(rules[arm])
#             positives[arm] += label
#             scores[arm] += score
#         n_samples[arm] += bs
#         means[arm] = positives[arm] / n_samples[arm]
#         mean_scores[arm] = scores[arm] / n_samples[arm]

#     def update_bounds_beam(t):
#         # bs = beam_size + (means < a).sum()
#         non_cause_ids = set(np.argwhere(means >= a).flatten())
#         sorted_rule_ids = sorted(range(n_arms), key = lambda i: mean_scores[i])
#         sorted_non_cause_ids = np.array([i for i in sorted_rule_ids if i in non_cause_ids])
        
#         beam_ids = sorted_non_cause_ids[:bs]
#         non_beam_ids = sorted_non_cause_ids[bs:]
#         if not beam_ids.size or not non_beam_ids.size: return 0
#         for i in beam_ids:
#             score_ub[i] = dup_bernoulli(mean_scores[i], beta / n_samples[i])
#         for i in non_beam_ids:
#             score_lb[i] = dlow_bernoulli(mean_scores[i], beta / n_samples[i])
            
#         ut = beam_ids[np.argmax(score_ub[beam_ids])]
#         lt = non_beam_ids[np.argmin(score_lb[non_beam_ids])]
#         B = score_ub[ut] - score_lb[lt]
#         if B >= beam_eps:
#             action_arm(ut)
#             action_arm(lt)
#         return B

#     def update_bounds_non_cause(t):
#         ids = np.argwhere(means >= a).flatten()
#         # print(f"non cause: {ids=}")
#         for i in ids:
#             lb[i] = dlow_bernoulli(means[i], beta / n_samples[i])
#         if not ids.size: return 0
#         lt = ids[np.argmin(lb[ids])]
#         B = a - lb[lt]
#         if B >= non_cause_esp:
#             action_arm(lt)
#         return B

#     def update_bounds_cause(t):
#         ids = np.argwhere(means < a).flatten()
#         # print(f"cause: {ids=}")
#         for i in ids:
#             ub[i] = dup_bernoulli(means[i], beta / n_samples[i])
#         if not ids.size: return 0
#         ut = ids[np.argmax(ub[ids])]
#         B = ub[ut] - a
#         if B >= cause_eps:
#             action_arm(ut)
#         return B
    
#     # Initialization
#     beam_bound = 1
#     cause_bound = 1
#     non_cause_bound = 1
#     for arm in tqdm(range(n_arms), disable=not verbose):
#         action_arm(arm, True)
#     it = 1
#     # Loop
#     with tqdm(total=n_arms * max_iter, disable=not verbose) as pbar:
#         while n_samples.sum() < n_arms * max_iter:
#             # Stop condition
#             if beam_bound <= beam_eps and cause_bound <= cause_eps and non_cause_bound <= non_cause_esp: 
#                 if verbose > 1: 
#                     print(f"Success: {beam_bound=:.4f} / {cause_bound=:.4f} / {non_cause_bound=:.4f})")
#                 break
#             if cause_bound <= cause_eps and non_cause_bound <= non_cause_esp and beam_size + (means < a).sum() >= n_arms:
#                 if verbose > 1:
#                     print(f"All rules pass on to next state: {cause_bound=:.4f}, {non_cause_bound=:.4f}")
#                 break
                
#             # Update bounds
#             beta = compute_beta(n_arms, it)
#             beam_bound = update_bounds_beam(it)
#             cause_bound = update_bounds_cause(it)
#             non_cause_bound = update_bounds_non_cause(it)
#             pbar.n = n_samples.sum()
#             pbar.refresh()
#             it += 1
#         else:
#             # Render how much we fail if we fail to reach the bound
#             if verbose > 1: 
#                 print(f"Fail: {beam_bound=:.4f} / {cause_bound=:.4f} / {non_cause_bound=:.4f}")
            
#     if verbose > 2:
#         print(f"ub={ub.round(4)}")
#         print(f"lb={lb.round(4)}")
#         print(f"preds={means.round(2)}")
#         print(f"n_samples={n_samples}")
#     if lucb_infos is not None:
#         lucb_infos.append({
#             "n_calls": int(n_samples.sum())
#         })
#     outputs = [((n_sample, ub_i, lb_i), mean, mean_score) for \
#                    n_sample, ub_i, lb_i, mean, mean_score in \
#                     zip(
#                         n_samples.tolist(),
#                         ub.tolist(), 
#                         lb.tolist(), 
#                         means.tolist(), 
#                         mean_scores.tolist()
#                     )]
#     return outputs

In [10]:
# def lucb_nSMK_simulation(rules, u, n, N, lucb_params, nl=0):
#     sim = lambda rule: nSMK_simulation(rule, u, n, nl)
#     return lucb(sim, rules, **lucb_params)

In [11]:
# def get_nSMK_SCM(n, u, do_lucb=True, N=100, nl=.05, lucb_params={}):
#     variables = get_nSMK_variables(n)
#     s, _, _ = nSMK_simulation([], u, n, nl=0) # No noise for the instance
#     domains = ((0,1),)*(len(variables)-1)
#     if do_lucb:
#         simulation = lambda rules: lucb_nSMK_simulation(rules, u, n, N, lucb_params, nl)
#     else: 
#         simulation = lambda rules: avg_nSMK_simulation(rules, u, n, N, nl)
#     return {"variables": variables[:-1], "domains":domains,
#             "simulation": simulation, "instance":s}

In [12]:
# def compute_comparison(rules, avg_scores, lucb_scores, full_verbose=False):
#     count = 0
#     for rule, avg_s, lucb_s in zip(rules, avg_scores, lucb_scores):
#         is_in_cause = any([rule[0][0] in cause for cause in ref_causes])
#         order_correct = is_in_cause == (avg_s>lucb_s)
#         if full_verbose: print(variables[rule[0][0]], avg_s, round(lucb_s,2), order_correct)
#         count += order_correct
#     print("Proportion of correct order:", round(count/len(rules),2))

In [13]:
def make_one_lucb(rules, seed=1):
    np.random.seed(seed)
    out = lucb(simulation, rules, **lucb_params)
    lucb_scores = [elt[1] for elt in out]
    
    avg_out = np.array([[[simulation(rule)[1],simulation(rule)[2]] for _ in range(N)] for rule in rules])
    avg_scores = avg_out[:,:,0].mean(1)
    
    compute_comparison(rules, avg_scores, lucb_scores, full_verbose=False)
    
    sorted_ids = sorted(range(len(rules)), key=lambda i: [out[i][1], out[i][2]])
    beam = [rules[i] for i in sorted_ids[:bs]]
    next_rules = get_rules(beam, SCM_lucb["domains"], SCM_lucb["instance"], [])

    found_vars = {variables[i] for i in sorted_ids[:bs]}
    exp_vars = {variables[i] for i in set().union(*ref_causes)}
    # print("Found:", found_vars)
    # print("Exp:", exp_vars)
    # print("Correct:", found_vars & exp_vars)
    # print("Missed:", exp_vars - found_vars)
    # print("Added:", found_vars - exp_vars)
    found_naive = {variables[i] for i in np.argsort(avg_scores)[:bs]}
    print(f"Prop correct: lucb={len(found_vars & exp_vars)/bs:.2f} / naive={len(found_naive & exp_vars)/bs:.2f}")
    
    return out, lucb_scores, avg_scores, next_rules

In [14]:
def get_avg_out(rules, simulation, seed=None):
    if seed is not None: np.random.seed(seed)
    out = []
    for rule in rules:
        line = []
        for _ in range(N):
            sim_out = simulation(rule)
            line.append((sim_out[1],sim_out[2]))
        out.append(line)
    return np.array(out)

In [15]:
def compare_prop_corect(rules, nl, n_seeds=30):
    prop_lucb = []
    prop_naive = []
    simulation = lambda rule: nSMK_simulation(rule, u, n, nl)
    exp_vars = {variables[i] for i in set().union(*[elt[3] for elt in sbs_causes])}
    
    for seed in tqdm(range(n_seeds)):
        np.random.seed(seed)
        
        out = lucb(simulation, rules, **lucb_params|{"verbose":0})
        sorted_ids = sorted(range(len(rules)), key=lambda i: [out[i][1], out[i][2]])
        found_vars = {variables[i] for i in sorted_ids[:bs]}
        prop_lucb.append(len(found_vars & exp_vars)/bs)
        
        avg_out = get_avg_out(rules, simulation)
        avg_scores = avg_out[:,:,0].mean(1)
        found_naive = {variables[i] for i in np.argsort(avg_scores)[:bs]}
        prop_naive.append(len(found_naive & exp_vars)/bs)

    print(f"Prop lucb: {np.mean(prop_lucb):.2f}±{np.std(prop_lucb):.2f}")  
    print(f"Prop naive: {np.mean(prop_naive):.2f}±{np.std(prop_naive):.2f}")           

In [16]:
def get_data_n_attacker(n):
    with open("../results/base-exact/structured.json") as file:
        data = json.load(file)
    for data_attacker in data:
        if data_attacker["n_attacker"] == n:
            return data_attacker["results"]
    raise f"{n=} not found"

In [17]:
def get_ref(n=5, instance_id=0):
    data = json.load(open("../results/base-exact/structured.json"))
    for dat_attacker in data:
        if dat_attacker["n_attacker"] == n:
            break
    else:
        raise f"{n=} not found"
    exact_res = dat_attacker["results"]
    
    u = exact_res[instance_id]["context"]
    exact_causes = exact_res[instance_id]["causes"]
    exact_e = exact_res[instance_id]["rules"]
    return u, exact_causes, exact_e

In [18]:
def test_context_multi_seeds(n, n_seeds):
    exact_res = get_data_n_attacker(n)

    evals = []
    for instance_id, instance in enumerate(exact_res):
        u = instance["context"]
        exact_causes = instance["causes"]
        ref_causes = list(map(tuple,exact_causes))
        
        scm = get_SMK_SCM(n, u)
        output_det = beam_search(**scm.get_input(), max_steps=4, beam_size=bs,verbose=0)
        det_causes = [tuple(elt[3]) for elt in output_det]
        det_eval = evaluate_full(det_causes, ref_causes)
        # print(ref_causes, det_causes)
        if det_eval["F1"] < .05: continue 
        print(instance_id, end = " ")
        print(f"Deterministic: {det_eval["F1"]:.2f}", end=" / ")

        eval_list = []
        for seed in tqdm(range(n_seeds)):
            SCM_avg = get_avg_nSMK_SCM(n, u, N, nl)
            output_avg = beam_search(**SCM_avg.get_input(), max_steps=4, beam_size=bs,verbose=0, epsilon=eps)
            avg_causes = [tuple(elt[3]) for elt in output_avg]
            avg_eval = evaluate_full(avg_causes, ref_causes)
    
            SCM_lucb = get_lucb_nSMK_SCM(n, u, nl, lucb_params|{"verbose":0})
            output_lucb = beam_search(**SCM_lucb.get_input(), max_steps=4, beam_size=bs, verbose=0, epsilon=eps)
            lucb_causes = [tuple(elt[3]) for elt in output_lucb]
            lucb_eval = evaluate_full(lucb_causes, ref_causes)
            eval_list.append((avg_eval["F1"], lucb_eval["F1"]))
        eval_list = np.array(eval_list)
        print(f"Naive: {eval_list[:,0].mean():.2f}±{eval_list[:,0].std():.2f} / LUCB: {eval_list[:,1].mean():.2f}±{eval_list[:,1].std():.2f}")
        evals.append(eval_list.mean(0))
    evals = np.array(evals)
    print(f"Naive: {evals[:,0].mean():.2f}±{evals[:,0].std():.2f} / LUCB: {evals[:,1].mean():.2f}±{evals[:,1].std():.2f}")

In [19]:
def test_some_contexts(n, n_tests):
    
    exact_res = get_data_n_attacker(n)
    # instance_ids = np.random.choice(np.arange(len(exact_res)), n_tests, replace=False)
    instance_ids = np.arange(len(exact_res))
    np.random.shuffle(instance_ids)

    evals = []
    k = 0
    for instance_id in instance_ids:
        if k > n_tests: break
        np.random.seed(0)
        u = exact_res[instance_id]["context"]
        exact_causes = exact_res[instance_id]["causes"]
        ref_causes = list(map(tuple,exact_causes))
        
        scm = get_SMK_SCM(n, u)
        output_det = beam_search(**scm.get_input(), max_steps=4, beam_size=bs,verbose=0)
        det_causes = [tuple(elt[3]) for elt in output_det]
        det_eval = evaluate_full(det_causes, ref_causes)
        # print(ref_causes, det_causes)
        if det_eval["F1"] < .05: continue 
        k += 1
        print(instance_id, end = " ")
        print(f"Deterministic: {det_eval["F1"]:.2f}", end=" / ")
        
        SCM_avg = get_avg_nSMK_SCM(n, u, N, nl)
        output_avg = beam_search(**SCM_avg.get_input(), max_steps=4, beam_size=bs,verbose=0, epsilon=eps)
        avg_causes = [tuple(elt[3]) for elt in output_avg]
        avg_eval = evaluate_full(avg_causes, ref_causes)
        print(f"Naive: {avg_eval["F1"]:.2f}", end=" / ")

        SCM_lucb = get_lucb_nSMK_SCM(n, u, nl, lucb_params|{"verbose":0})
        output_lucb = beam_search(**SCM_lucb.get_input(), max_steps=4, beam_size=bs, verbose=0, epsilon=eps)
        lucb_causes = [tuple(elt[3]) for elt in output_lucb]
        lucb_eval = evaluate_full(lucb_causes, ref_causes)
        print(f"LUCB: {lucb_eval["F1"]:.2f}")
        # print("--"*40)
        evals.append((avg_eval["F1"], lucb_eval["F1"]))
    evals = np.array(evals)
    print(f"Naive: {evals[:,0].mean():.2f}±{evals[:,0].std():.2f} / LUCB: {evals[:,1].mean():.2f}±{evals[:,1].std():.2f}")

In [20]:
def test_context_multi_seed(n, instance_id, n_seeds):
    
    exact_res = get_data_n_attacker(n)
    u = exact_res[instance_id]["context"]
    exact_causes = exact_res[instance_id]["causes"]
    ref_causes = list(map(tuple,exact_causes))

    evals = []
    
    for seed in range(n_seeds):
        np.random.seed(seed)

        SCM_avg = get_avg_nSMK_SCM(n, u, N=N, nl=nl)
        output_avg = beam_search(**SCM_avg.get_input(), max_steps=4, beam_size=bs,early_stop=False,verbose=0, epsilon=eps)
        avg_causes = [tuple(elt[3]) for elt in output_avg]
        avg_eval = evaluate_full(avg_causes, ref_causes)
        print(f"Naive F1: {avg_eval["F1"]:.2f}", end=" / ")
        
        SCM_lucb = get_lucb_nSMK_SCM(n, u, nl, lucb_params|{"verbose":0})
        output_lucb = beam_search(**SCM_lucb.get_input(), max_steps=4, beam_size=bs, early_stop=False,verbose=0, epsilon=eps)
        lucb_causes = [tuple(elt[3]) for elt in output_lucb]
        lucb_eval = evaluate_full(lucb_causes, ref_causes)
        print(f"LUCB F1: {lucb_eval["F1"]:.2f}")
        # print("--"*40)
        evals.append((avg_eval["F1"], lucb_eval["F1"]))
    evals = np.array(evals)
    print(f"Naive: {evals[:,0].mean():.2f}±{evals[:,0].std():.2f} / LUCB: {evals[:,1].mean():.2f}±{evals[:,1].std():.2f}")

In [21]:
def interpolate(x, vmin=None, vmax=None, vrefs=None):
    # return x
    if vrefs is not None:
        vmin = np.min(vrefs)
        vmax = np.max(vrefs)
    else:
        if vmin is None:
            vmin = np.min(x)
        if vmax is None:
            vmax = np.max(x)
    
    if vmin == vmax: return x
    return (x - vmin) / (vmax - vmin)

# Main

Problem to solve: for 5/10 attackers, LUCB misses way more than naive sampling

In [22]:
n = 10

In [23]:
variables = get_SMK_V(n)

bs = 12
max_iter = 20
# N = 100
eps = .65
nl = 2
frac_init = .8
frac_remaining = 1
lucb_params = {"beam_size": bs, "a": eps, "cause_eps": .1, 
               "beam_eps": .1, "max_iter": max_iter, "verbose": 2, 
               "batch_size": int(max_iter*(1-frac_init)/frac_remaining), 
               "init_batch_size": int(max_iter*frac_init),
               # "batch_size": 1, 
               # "init_batch_size": 5,
               "non_cause_eps": .1}

## Full general test

In [24]:
# u, exact_causes, exact_e = get_ref(n, 0)
# ref_causes = list(map(tuple,exact_causes))
# var_cause = np.array(list({var for cause in ref_causes for var in cause}))

In [25]:
# # Compute

# # Build rules and refs
# scm = get_SMK_SCM(n, u)
# rules_1 = get_initial_rules(scm.v[:-1],scm.D[:-1])
# rules_2 = get_rules(rules_1, scm.D[:-1], scm.v[:-1], [])
# rules = rules_2
# x = np.arange(len(rules))
# ref_out = scm.apply_interventions(rules)
# ref_psi = np.array([elt[2]/len(scm.V) for elt in ref_out])
# ref_phi = np.array([elt[1] for elt in ref_out])
# sorted_ids_phi = np.argsort(ref_phi)
# sorted_ids_psi = np.argsort(ref_psi)

# # Compute ref stochastic
# # scm_avg_large = get_avg_nSMK_SCM(n, u, 2000, nl)
# # out_avg_large = scm_avg_large.apply_interventions(rules)
# # avg_large_psi = np.array([elt[2]/len(scm_avg_large.V) for elt in out_avg_large])
# # avg_large_phi = np.array([elt[1] for elt in out_avg_large])

# # Compute average
# scm_avg = get_avg_nSMK_SCM(n, u, max_iter, nl)
# out_avg = scm_avg.apply_interventions(rules)
# avg_psi = np.array([elt[2]/len(scm_avg.V) for elt in out_avg])
# avg_phi = np.array([elt[1] for elt in out_avg])

# # Compute lucb
# scm_lucb = get_lucb_nSMK_SCM(n, u, nl,lucb_params|{"verbose":2})
# out_lucb = scm_lucb.apply_interventions(rules)
# lucb_psi = np.array([elt[2] for elt in out_lucb])
# lucb_phi = np.array([elt[1] for elt in out_lucb])
# lucb_iter = np.array([elt[0][0] for elt in out_lucb])

# print(f"actual ref psi: {sorted(ref_psi)[bs]:.2f}")
# print(f"avg ref psi: {sorted(avg_psi)[bs]:.2f}")

In [26]:
# avg_v = np.mean([nSMK_model(u, [], n, nl/len(scm.V)) for _ in range(lucb_params["max_iter"])], axis=0)
# (float(sum(avg_v)) - 1)/len(scm.V), (sum(scm.v) - 1)/len(scm.V)

In [27]:
# plt.plot(sorted(avg_psi))
# plt.plot(sorted(ref_psi))
# plt.scatter(x, lucb_psi[sorted_ids_psi])
# plt.vlines(bs, 0,1)

In [28]:
# a = [np.random.rand() < .2 for _ in range(10000)]
# np.mean(a)

In [29]:
# _, axes = plt.subplots(1,3,figsize=(15,5))
# max_x = -1

# # sorted_ids_psi = np.argsort(avg_large_psi)
# # sorted_ids_phi = np.argsort(avg_large_psi)
# plot_ids_psi = [i for i in sorted_ids_psi if ref_phi[i] == 1][:max_x]

# def filter_psi(values):
#     values = interpolate(values[plot_ids_psi])
#     return np.arange(len(values)), values

# alpha = .5
# # Plot refs
# axes[0].plot(*filter_psi(ref_psi), alpha=alpha)#, c=c)
# axes[1].plot(ref_phi[sorted_ids_phi][:max_x], alpha=alpha)#, c=c)

# axes[0].vlines(bs, 0, .7)
# axes[1].hlines(eps, 0, len(rules[:max_x]))

# # Average
# # avg_colors = np.array(["orange" if value < sorted(avg_psi[plot_ids_psi])[bs] else "purple" for value in avg_psi])
# axes[0].scatter(*filter_psi(avg_psi), alpha=alpha, label="avg psi")#, c=avg_colors[plot_ids_psi])
# axes[1].scatter(x[:max_x], avg_phi[sorted_ids_phi][:max_x], alpha=alpha, label="avg phi")

# # axes[0].plot(interpolate(avg_large_psi[sorted_ids_psi])[:max_x], label="large")
# # axes[1].plot(avg_large_psi[sorted_ids_phi][:max_x], label="large")

# # LUCB
# # lucb_colors = np.array(["black" if value < sorted(lucb_psi[plot_ids_psi])[bs] else "red" for value in lucb_psi])
# axes[0].scatter(*filter_psi(lucb_psi), alpha=alpha, label="lucb psi")#, c=lucb_colors[plot_ids_psi])
# axes[1].scatter(x[:max_x], lucb_phi[sorted_ids_phi][:max_x], alpha=alpha, label="lucb phi")
# axes[2].scatter(x[:max_x], lucb_iter[sorted_ids_phi][:max_x], alpha=alpha)
# axes[0].hlines(interpolate(sorted(lucb_psi)[bs], vrefs=lucb_psi), 0, len(rules[:max_x]), alpha=alpha)

# # axes[0].plot(sorted(lucb_psi))
# # lucb_threshold = sorted(lucb_psi)[bs]
# # axes[0].hlines(lucb_threshold, 0,len(rules))
# # axes[0].scatter(sorted_ids_psi[lucb_threshold],[lucb_psi[lucb_threshold]])

# # axes[0].set_xscale('log')
# axes[0].legend()
# # axes[1].set_xscale('log')
# axes[1].legend()
# # axes[2].set_xscale('log')

## General tests

In [ ]:
test_context_multi_seeds(n, n_seeds)

6 Deterministic: 0.24 / 

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
# evals = test_some_contexts(n, 50)

In [ ]:
# test_context_multi_seed(n, 17, 10)

## Single overall test

In [ ]:
# u, exact_causes, exact_e = get_ref(n=n, instance_id=12)

In [ ]:
# print(f"Found {len(exact_causes)} causes")
# print(f"Size of the interventions: {[list(map(len,exact_e))]}")

In [ ]:
# ref_causes = list(map(tuple,exact_causes))

In [ ]:
# list(map(lambda C: var2label(C, variables), ref_causes))

In [ ]:
# SCM_avg = get_avg_nSMK_SCM(n, u, N, nl)
# SCM_lucb = get_lucb_nSMK_SCM(n, u, nl, lucb_params)
# scm = get_SMK_SCM(n, u)

In [ ]:
# np.random.seed(42)
# output_avg = beam_search(**SCM_avg.get_input(), max_steps=4, beam_size=bs,verbose=2, epsilon=eps)
# print("\nRESULTS\n")
# show_rules(output_avg, SCM_avg.V)

In [ ]:
# np.random.seed(42)
# output_lucb = beam_search(**SCM_lucb.get_input(), max_steps=4, beam_size=bs,verbose=2, epsilon=eps)
# print("\nRESULTS\n")
# show_rules(output_lucb, SCM_lucb.V)

In [ ]:
# avg_causes = [tuple(elt[3]) for elt in output_avg]
# lucb_causes = [tuple(elt[3]) for elt in output_lucb]
# avg_F1 = evaluate_full(avg_causes, ref_causes)["F1"]
# lucb_F1 = evaluate_full(lucb_causes, ref_causes)["F1"]
# print(f"Naive: {avg_F1:.2f} / LUCB: {lucb_F1:.2f}")